## Test the instances with EnergyCommunity.jl

This notebook aims to obtain the results of the instances with EnergyCommunity.jl.

Each instance is defined by:
- `configuration file`: configuration file of the instance, which contains the parameters of the model. The configuration file is a YAML file, e.g. `energy_community_model.yaml`
- `data file(s)`: data file of the instance, which contains the data of the model. The data file(s) are CSV files, e.g. `input_resources.csv` as defined in the configuration file under general->optional_datasets

For more details on the configuration files, please see the [documentation of EnergyCommunity.jl](https://spsunipi.github.io/EnergyCommunity.jl/dev/configuration/configuration/).

This notebook is structured as follows:
1. Imports the necessary packages
2. Defines the path to the configuration file
3. Create the model and solve it
4. Print the results

To use this notebook, you can use jupyter nootebooks or any other IDE that supports Julia, such as VS Code.
We recommend to use this notebook using VS Code with its Julia extension, which allows to run the code cells and visualize the results in a more interactive way.

### 1. Imports the necessary packages

In [ ]:
# Package manager to setup the environment
import Pkg
Pkg.activate(".")
Pkg.instantiate()  # optional; comment after first execution

### 2. Defines the path to the configuration file

In [ ]:
# Load the data
fconfig = "energy_community_model_new.yml"

# define if network is stochastic: if the configuration file contains
# "_sto.yml" it is considered stochastic, otherwise it is deterministic
is_stochastic = occursin("_sto.yml", fconfig)

# Ensure environment is set up with the correct version of
# EnergyCommunity.jl (the stochastic branch is on the `stochastic` rev)
if is_stochastic
    Pkg.add(url="https://github.com/SPSUnipi/EnergyCommunity.jl", rev="stochastic")
else
    Pkg.add(url="https://github.com/SPSUnipi/EnergyCommunity.jl", rev="main")
end

In [ ]:
using EnergyCommunity
using Gurobi  # Commercial solver; if you don't have a license you can use HiGHS
using HiGHS
using JuMP
using Random

# Use the same RNG seed as `csv2nc4.jl` so that the (s, eps) scenario
# draws coincide between the SMS++ TSSB pipeline and this script —
# necessary for the FO comparison to be meaningful.
Random.seed!(123)

# default solver
optimizer = Gurobi.Optimizer  # if you have a Gurobi license, otherwise use HiGHS
# optimizer = HiGHS.Optimizer  # Uncomment this line to use HiGHS instead of Gurobi

### 3. Create the model and solve it

In [ ]:
obj_value = nothing
optimal_design = nothing

if is_stochastic
    println("The model is stochastic.")

    # Mirrors the upstream `RunStochModel(TBD).jl` example of
    # https://github.com/SPSUnipi/EnergyCommunity.jl branch `stochastic`,
    # so the YAML must follow that schema (`general.n_s`, `general.n_eps`,
    # `general.uncertain_var`, per-component `profile.std`, single
    # `market.profile`).
    data = read_input(fconfig)
    (gen_data, users_data, market_data) = explode_data(data)

    scen_s_sample = field(gen_data, "n_s")
    scen_eps_sample = field(gen_data, "n_eps")
    unc_var = field(gen_data, "uncertain_var")

    sigma_load = get(gen_data, "sigma_load", 0.3)
    mean_pv    = get(gen_data, "mean_pv",    1.0)
    sigma_pv   = get(gen_data, "sigma_pv",   0.1)
    mean_wind  = get(gen_data, "mean_wind",  0.95)
    sigma_wind = get(gen_data, "sigma_wind", 0.15)

    (point_s_load, point_s_pv, point_s_wind, scen_probability) =
        pem_extraction(scen_s_sample, sigma_load,
                       mean_pv, sigma_pv,
                       mean_wind, sigma_wind,
                       unc_var)

    sampled_scenarios = scenarios_generator(
        data,
        point_s_load, point_s_pv, point_s_wind,
        scen_s_sample, scen_eps_sample, unc_var;
        point_probability=scen_probability,
        first_stage=true,
        deterministic=(scen_s_sample == 1 && scen_eps_sample == 1),
    )

    model = StochasticEC(fconfig, EnergyCommunity.GroupCO(), optimizer,
                         sampled_scenarios, scen_s_sample, scen_eps_sample)
    build_specific_model!(EnergyCommunity.GroupCO(), model, optimizer)

    set_parameters_ECmodel!(model, 1e-2, 60 * 60, Threads.nthreads(), 1)
    optimize_deterministic_ECmodel(model)

    obj_value = objective_value(model.model)
    optimal_design = model.results[:x_us].data
else
    println("The model is deterministic.")

    model = ModelEC(fconfig, EnergyCommunity.GroupCO(), optimizer)
    build_model!(model)
    optimize!(model)

    obj_value = objective_value(model)
    optimal_design = value.(model.results[:x_us])
end

### 4. Print the results

In [ ]:
println("Optimal value: ", obj_value)
println("Optimal installed capacity by user: ", optimal_design)